# Fine-tuning wav2vec 2.0 for deep fake detection

In [1]:
import numpy as np
import random

import evaluate
from datasets import load_dataset, Audio
from transformers import (
    AutoFeatureExtractor, 
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

from IPython.display import Audio, display

from huggingface_hub import notebook_login
notebook_login()

Token has not been saved to git credential helper.


In [2]:
max_duration = 30.0
data_repo = "ArissBandoss/fake_or_real_dataset_for_norm"
model_id = "facebook/wav2vec2-base"

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

### I. Dataset

In [3]:
deepfake_data = load_dataset(data_repo)
deepfake_data = deepfake_data["train"].train_test_split(
    seed=42, 
    shuffle=True, 
    test_size=0.3,
)

print(deepfake_data["train"][0])

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

{'audio': {'path': 'file5135.mp3.wav_16k.wav_norm.wav_mono.wav_silence.wav', 'array': array([ 0.04669189,  0.05059814,  0.04315186, ..., -0.02218628,
       -0.02243042, -0.021698  ]), 'sampling_rate': 16000}, 'label': 1, 'source': 'Fake-or-Real (FoR)'}


### II. Preprocessing

In [4]:
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, 
    do_normalize=True, 
    return_attention_mask=True
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/configuration_utils.py:312: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [5]:
# obtaining the model’s sample rate from the feature extractor
sampling_rate = feature_extractor.sampling_rate
print(sampling_rate)

# resample the dataset using the cast_column() method and Audio feature from 🤗 Datasets
#deepfake_data = deepfake_data.cast_column("audio", Audio(sampling_rate=sampling_rate))
deepfake_data

16000


DatasetDict({
    train: Dataset({
        features: ['audio', 'label', 'source'],
        num_rows: 48510
    })
    test: Dataset({
        features: ['audio', 'label', 'source'],
        num_rows: 20790
    })
})

In [6]:
sample = deepfake_data["train"][0]["audio"]
display(sample)

# compute the mean and variance of our raw audio data
print(f"Mean: {np.mean(sample['array']):.3}, Variance: {np.var(sample['array']):.3}")

# apply the feature extractor and see what the outputs look like
inputs = feature_extractor(sample["array"], sampling_rate=sample["sampling_rate"])
print(f"inputs keys: {list(inputs.keys())}")

print(
    f"Mean: {np.mean(inputs['input_values']):.3}, Variance: {np.var(inputs['input_values']):.3}"
)

{'path': 'file5135.mp3.wav_16k.wav_norm.wav_mono.wav_silence.wav',
 'array': array([ 0.04669189,  0.05059814,  0.04315186, ..., -0.02218628,
        -0.02243042, -0.021698  ]),
 'sampling_rate': 16000}

Mean: 0.0273, Variance: 0.0166
inputs keys: ['input_values', 'attention_mask']
Mean: -1.37e-08, Variance: 1.0


In [7]:
deepfake_data_encoded = deepfake_data.map(
    preprocess_function,
    remove_columns=["audio", "source"],
    batched=True,
    batch_size=100,
    num_proc=4,
)
deepfake_data_encoded

DatasetDict({
    train: Dataset({
        features: ['label', 'input_values', 'attention_mask'],
        num_rows: 48510
    })
    test: Dataset({
        features: ['label', 'input_values', 'attention_mask'],
        num_rows: 20790
    })
})

In [8]:
labels = deepfake_data["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

id2label["0"]

'real'

### II. Training

In [9]:
num_labels = len(id2label)

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
)

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
model_name = model_id.split("/")[-1]
batch_size = 16
gradient_accumulation_steps = 1
num_train_epochs = 5

training_args = TrainingArguments(
    f"{model_name}-deepfake-detection",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    push_to_hub=True,
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/training_args.py:1573: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [11]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [12]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=deepfake_data_encoded["train"],
    eval_dataset=deepfake_data_encoded["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipykernel_128743/3094494697.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
kwargs = {
    "dataset_tags": "ArissBandoss/fake_or_real_dataset_for_norm",
    "dataset": "fake_or_real_dataset_for_norm",
    "model_name": f"{model_name}-deepfake-detection",
    "finetuned_from": model_id,
    "tasks": "audio-classification",
}

# The training results can now be uploaded to the Hub. To do so, execute the .push_to_hub command:
trainer.push_to_hub(**kwargs)

### III. Testing the model

In [ ]:
"""pipe = pipeline(
    "audio-classification", model=""
)"""